# BASELINES - Previsione della temperatura

Primo modello di previsione della **temperatura media giornaliera** (`t2m_mean`) di Foggia, confrontato con tre baseline naïve.

**L'idea in breve.** Non prevediamo la temperatura direttamente, ma di **quanto si discosterà dal normale**. Il "normale" di un giorno è la sua climatologia (la temperatura media di quel giorno del calendario negli anni di train), corretta per il riscaldamento recente. Tolto il ciclo stagionale, quello che resta segue quasi perfettamente un modello lineare **AR(2)**, come indicato dalla PACF del residuo in `data_analysis.ipynb`.

**Problema di previsione.** Alla fine del giorno *t*, usando solo dati fino a *t*, si prevede la temperatura del giorno *t + h*, per gli orizzonti **h = 1, 3, 7, 14 giorni**. Si stima un modello separato per ogni orizzonte (strategia *diretta*).

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_USER = "SinghSahiljit44"  # Es. sahiljit
    REPO_NAME = "Weather_forecasting"
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

    # Clona la repository se non è già stata clonata nella sessione
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

    # Cambia la directory di lavoro nella root del progetto
    os.chdir(REPO_NAME)
    
    # Installa le dipendenze necessarie nell'ambiente di Colab
    %pip install -q -r requirements.txt
    
    PROJECT_ROOT = Path.cwd()
else:
    # --- CONFIGURAZIONE LOCALE  ---
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f" [OK] Ambiente rilevato: {'Google Colab' if IN_COLAB else 'Locale'}")
print(f" [OK] Directory di lavoro impostata a: {Path.cwd()}")

 [OK] Ambiente rilevato: Locale
 [OK] Directory di lavoro impostata a: /home/matteo/Documents/GitHub/Weather_forecasting


## 1. Dati e split temporale

| Variabile | Significato |
|---|---|
| `y` | Serie da prevedere: temperatura media giornaliera a 2 m, in °C |
| `HORIZONS` | Orizzonti di previsione, in giorni |
| `train` | 1990-2017: qui si stimano climatologia e modello |
| `val` | 2018-2021: qui si confrontano i modelli |
| `test` | 2022-2025: **non usato in questo notebook**, si tiene per la valutazione finale |

Lo split è cronologico, mai casuale: in una serie temporale il modello non deve vedere il futuro.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

from src.data.preprocessing import load_daily

TARGET = "t2m_mean"
HORIZONS = [1, 3, 7, 14]
TRAIN_END = 2017  # train: 1990-2017
VAL_END = 2021    # validation: 2018-2021, test: 2022-2025 (not used here)

df = load_daily().asfreq("D") # force daily frequency
y = df[TARGET]

year = y.index.year
train = year <= TRAIN_END
val = (year > TRAIN_END) & (year <= VAL_END)
test = year > VAL_END

print(f"Train: {train.sum()} giorni | Validation: {val.sum()} giorni | Test: {test.sum()} giorni (non usato)")

Train: 10227 giorni | Validation: 1461 giorni | Test: 1460 giorni (non usato)


## 2. Climatologia e anomalie

| Variabile | Significato |
|---|---|
| `clim` | **Climatologia**: per ogni giorno del calendario (es. 15 marzo), la temperatura media di quel giorno negli anni di **train**, lisciata su ±15 giorni per togliere il rumore dovuto ai soli 28 anni disponibili |
| `anom` | **Anomalia**: `y - clim`, cioè quanto un giorno è più caldo (+) o più freddo (−) del normale |
| `anom_365` | **Anomalia media degli ultimi 365 giorni**: misura di quanto l'ultimo anno è stato più caldo del normale. Porta dentro il **trend** di riscaldamento |

La climatologia si calcola **solo sul train**: usare anche validation e test sarebbe leakage.

Il secondo numero stampato mostra il problema del trend: la validation è in media **0,84 °C più calda** della climatologia del train.

In [8]:
SMOOTH_DAYS = 31  # +-15 days around each calendar day


def climatology(series, mask, window=SMOOTH_DAYS):
    key = series.index.strftime("%m-%d")
    by_day = series[mask].groupby(key[mask]).mean()
    half = window // 2
    padded = np.r_[by_day.values[-half:], by_day.values, by_day.values[:half]]
    smooth = pd.Series(np.convolve(padded, np.ones(window) / window, mode="valid"), index=by_day.index)
    return pd.Series(smooth.reindex(key).to_numpy(), index=series.index)


clim = climatology(y, train)          # normal temperature of each calendar day (train only)
anom = y - clim                       # how much warmer/colder than normal
anom_365 = anom.rolling(365).mean()   # mean anomaly of the last 365 days: carries the trend

# on the validation set, the mean anomaly is +0.84 °C, which is a sign of global warming
print(f"Anomalia media - train: {anom[train].mean():+.2f} °C | validation: {anom[val].mean():+.2f} °C")

Anomalia media - train: +0.00 °C | validation: +0.84 °C


Per ogni orizzonte *h* servono due maschere, `fit` e `evaluate`: si usano solo le origini il cui giorno da prevedere (*t + h*) cade nello stesso periodo. Ad esempio, il 31/12/2017 con h = 7 non entra nel train, perché il suo target è nel 2018.

In [9]:
def fit_eval_masks(h):
    target_year = (y.index + pd.Timedelta(days=h)).year
    fit = train & (target_year <= TRAIN_END)
    evaluate = val & (target_year <= VAL_END)
    return fit, evaluate

## 3. Baseline naïve

Riferimenti da battere. Tutte le previsioni sono indicizzate sul giorno di origine *t*.

| Baseline | Formula | Idea |
|---|---|---|
| **Persistenza** | ŷ(t+h) = y(t) | "Domani come oggi" |
| **Climatologia (train)** | ŷ(t+h) = clim(t+h) | "Il normale di quel giorno" |
| **Climatologia aggiornata** | ŷ(t+h) = clim(t+h) + anom_365(t) | Il normale, alzato del riscaldamento dell'ultimo anno |

In [ ]:
def baseline_predictions(h):
    return {
        "Persistenza": y,
        "Climatologia (train)": clim.shift(-h),
        "Climatologia aggiornata": clim.shift(-h) + anom_365,
    }

## 4. Modello AR(2) con regressione lineare

$$\hat{y}(t+h) = \underbrace{clim(t+h) + anom\_365(t)}_{\text{climatologia aggiornata}} + \beta_0 + \beta_1 \cdot anom\_t + \beta_2 \cdot anom\_t\_minus\_1$$

La regressione lineare non prevede la temperatura, ma solo la **correzione** da aggiungere alla climatologia aggiornata.

| Variabile | Significato |
|---|---|
| `anom_t` | Anomalia di **oggi** rispetto alla climatologia aggiornata |
| `anom_t_minus_1` | Anomalia di **ieri** rispetto alla climatologia aggiornata |
| `reference` | Climatologia aggiornata del giorno *t + h*: il punto di partenza della previsione |
| `residual_target` | Valore da imparare: temperatura vera di *t + h* meno `reference` |
| `ar2_models[h]` | Regressione lineare stimata sul train per l'orizzonte *h* |
| `ar2_predictions[h]` | Previsione finale in °C: `reference` + correzione del modello |

**Perché le anomalie e non la temperatura?** Il ciclo stagionale non è una retta: una regressione lineare sulla temperatura grezza non può rappresentarlo. Tolta la climatologia, la dinamica residua è lineare. Riferire tutto alla climatologia *aggiornata* gestisce inoltre il trend: il modello non deve imparare il riscaldamento, che gli arriva già da `anom_365`.

In [ ]:
features = pd.DataFrame(
    {
        "anom_t": anom - anom_365,                   # today's anomaly vs the updated climatology
        "anom_t_minus_1": anom.shift(1) - anom_365,  # yesterday's anomaly vs the updated climatology
    }
)

ar2_models = {}
ar2_predictions = {}
for h in HORIZONS:
    reference = clim.shift(-h) + anom_365       # updated climatology of day t+h
    residual_target = y.shift(-h) - reference   # what the model has to learn
    usable = features.notna().all(axis=1) & residual_target.notna()
    fit, _ = fit_eval_masks(h)

    model = LinearRegression().fit(features[usable & fit], residual_target[usable & fit])
    prediction = pd.Series(np.nan, index=y.index)
    prediction[usable] = reference[usable] + model.predict(features[usable])

    ar2_models[h] = model
    ar2_predictions[h] = prediction

## 5. Valutazione sulla validation

| Metrica | Significato |
|---|---|
| **MAE** | Errore assoluto medio, in °C: quanto si sbaglia in media |
| **RMSE** | Radice dell'errore quadratico medio: penalizza di più gli errori grandi |
| **bias** | Errore medio con segno (previsto − vero): negativo = il modello prevede troppo freddo |
| **skill** | `1 − MSE_modello / MSE_climatologia_aggiornata`: 0 = come la climatologia aggiornata, 1 = perfetto, negativo = peggio |

Tutti i modelli sono valutati sugli stessi giorni.

In [ ]:
def scores(prediction, h, mask):
    error = (prediction - y.shift(-h))[mask]
    return {"MAE": error.abs().mean(), "RMSE": np.sqrt((error**2).mean()), "bias": error.mean()}


rows = []
for h in HORIZONS:
    _, evaluate = fit_eval_masks(h)
    mask = evaluate & ar2_predictions[h].notna() & y.shift(-h).notna()
    candidates = baseline_predictions(h) | {"AR(2) regressione lineare": ar2_predictions[h]}
    reference_mse = scores(candidates["Climatologia aggiornata"], h, mask)["RMSE"] ** 2
    for name, prediction in candidates.items():
        s = scores(prediction, h, mask)
        s["skill vs clim. aggiornata"] = 1 - s["RMSE"] ** 2 / reference_mse
        rows.append({"modello": name, "h": h, **s})

results = pd.DataFrame(rows)
order = list(candidates)
for metric in ["MAE", "bias", "skill vs clim. aggiornata"]:
    print(f"\n{metric} sulla validation 2018-2021")
    print(results.pivot(index="modello", columns="h", values=metric).loc[order].round(3).to_string())

### Commento ai risultati

| MAE (°C) | h = 1 | h = 3 | h = 7 | h = 14 |
|---|---|---|---|---|
| Persistenza | 1,25 | 2,23 | 2,74 | 3,23 |
| Climatologia (train) | 2,19 | 2,19 | 2,19 | 2,19 |
| Climatologia aggiornata | 2,09 | 2,10 | 2,11 | 2,11 |
| **AR(2)** | **1,16** | **1,89** | **2,08** | **2,11** |

- **L'AR(2) è il migliore a tutti gli orizzonti.** A 1 giorno sbaglia in media di 1,16 °C, con uno skill di 0,68 rispetto alla climatologia aggiornata.
- **Il vantaggio svanisce con l'orizzonte**: skill 0,68 a 1 giorno, 0,18 a 3, 0,03 a 7, circa zero a 14. Oltre una settimana la previsione migliore è il "normale recente": è il limite di predicibilità con soli dati locali.
- **La persistenza è buona solo a 1 giorno** e poi peggiora rapidamente: a 14 giorni sbaglia più della climatologia.
- **Il trend conta**: la climatologia del solo train ha un bias di −0,84 °C, cioè prevede sistematicamente troppo freddo. La correzione con `anom_365` lo riduce a −0,07 °C, e l'AR(2) a circa −0,03 °C.

In [ ]:
coefficients = pd.DataFrame(
    {h: [m.intercept_, *m.coef_] for h, m in ar2_models.items()},
    index=["intercetta", "anom_t", "anom_t_minus_1"],
).T
coefficients.index.name = "h"
coefficients.round(3)

### Interpretazione dei coefficienti

Per h = 1 il modello stimato è circa:

$$\text{anomalia di domani} \approx 1{,}03 \cdot \text{anomalia di oggi} - 0{,}25 \cdot \text{anomalia di ieri}$$

- **`anom_t` ≈ 1,03: inerzia.** Se oggi è più caldo del normale, domani lo sarà quasi altrettanto.
- **`anom_t_minus_1` ≈ −0,25: rientro.** A parità di oggi, se l'anomalia dura già da ieri tende a esaurirsi.
- **L'intercetta è quasi zero** (0,004 °C): la climatologia aggiornata è già centrata, il modello non deve correggerla.

Al crescere di *h* il peso di `anom_t` scende (1,03 → 0,51 → 0,17 → 0,04): l'informazione di oggi si consuma in circa una settimana, coerentemente con l'ACF del residuo.

In [ ]:
start, end = "2021-01-01", "2021-04-30"
h = 1
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y[start:end], color="black", linewidth=1.2, label="Osservata")
# predictions are indexed by origin t: shift by h to align them with the day they forecast
ax.plot(ar2_predictions[h].shift(h)[start:end], color="tab:blue", label=f"AR(2), h={h}")
ax.plot((clim + anom_365.shift(h))[start:end], color="tab:orange", label="Climatologia aggiornata")
ax.set_title(f"Temperatura media giornaliera - previsione a {h} giorno (validation, inizio 2021)")
ax.set_ylabel("°C")
ax.legend()
plt.tight_layout()
plt.show()

### Esempio: gennaio-aprile 2021

La previsione a 1 giorno (blu) segue le oscillazioni della temperatura osservata (nero) con circa un giorno di ritardo: è il comportamento tipico di un modello autoregressivo, che reagisce a un cambiamento solo dopo averlo visto. La climatologia aggiornata (arancione) dà solo l'andamento stagionale.

## Conclusioni

- La baseline da battere per i modelli successivi (XGBoost, LSTM) è l'**AR(2)**: MAE **1,16 °C a 1 giorno, 1,89 a 3, 2,08 a 7** sulla validation.
- Un modello che a 7 giorni scende molto sotto 2,0 °C va controllato per **leakage** (climatologia o scaler calcolati su tutti gli anni, componenti STL usate come feature).
- Possibili miglioramenti, da provare in `feature_engineering.ipynb`: le altre variabili del dataset (Tmin/Tmax, temperatura e umidità del suolo), sempre riferite alla climatologia aggiornata.